# Medical NER — chạy end-to-end trên Kaggle (GPU 16GB)

**TRƯỚC KHI `Run All`:**
1. Panel bên phải → **Settings → Accelerator → GPU T4 x2** (hoặc P100).
2. **Settings → Internet → ON** (cần clone repo, tải model, gọi RxNav API).

Sau khi chạy xong, tải **`output_ner.zip`** ở panel *Output* bên phải để nộp thi.


In [ ]:
# 1) Clone repo
import os
REPO = "https://github.com/Khanhhh239/Medical_Retrieve.git"
D = "/kaggle/working/Medical_Retrieve"
if not os.path.isdir(D):
    !git clone -q {REPO} {D}
else:
    !cd {D} && git pull -q
%cd {D}


In [ ]:
# 2) Cài deps (torch + transformers có sẵn trên Kaggle)
!pip install -q rapidfuzz pyyaml regex seqeval
import torch, transformers
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("transformers", transformers.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


In [ ]:
# 3) Test nhanh (đảm bảo code lành)
!python -m pytest -q 2>&1 | tail -3


In [ ]:
# 4) XÂY KHO MÃ — RxNav API (thuốc) + tải ICD-10 (bệnh). Cần Internet ON.
!python scripts/build_kb.py --rxnorm --icd


In [ ]:
# 5) Sinh data huấn luyện (nhiều hơn để chất lượng cao)
!python scripts/gen_synth.py --n_train 12000 --n_dev 500


In [ ]:
# 6) TRAIN NER — Kaggle 16GB: batch lớn + AdamW + seq dài (KHÔNG OOM)
!python scripts/train_ner.py --model xlm-roberta-base --epochs 4 \
  --batch_size 16 --grad_accum 1 --max_length 384 --optim adamw_torch \
  --out models/ner
# Muốn CHẤT LƯỢNG CAO HƠN (nếu GPU 16GB cho phép): xlm-roberta-large
# !python scripts/train_ner.py --model xlm-roberta-large --epochs 3 \
#   --batch_size 8 --grad_accum 2 --max_length 384 --optim adamw_torch --out models/ner


In [ ]:
# 7) Inference bằng LUẬT (rule) -> output.zip
!python scripts/predict.py --pipeline baseline --out_dir output --zip


In [ ]:
# 8) Inference bằng MODEL (NER) -> output_ner.zip  (bản nộp chính)
!python scripts/predict.py --pipeline ner --model_dir models/ner --out_dir output_ner --zip


In [ ]:
# 9) Đo trên dev synthetic (so luật vs model)
!python scripts/evaluate.py --gold data/synthetic/dev.jsonl --pipeline baseline+link
!python scripts/evaluate.py --gold data/synthetic/dev.jsonl --pipeline "ner:models/ner"


In [ ]:
# 10) So sánh 1 file VĂN XUÔI (8.txt) — luật vs model
import json
r = json.load(open("output/8.json")); n = json.load(open("output_ner/8.json"))
print("RULE 8.json  :", json.dumps(r, ensure_ascii=False)[:400])
print("MODEL 8.json :", json.dumps(n, ensure_ascii=False)[:400])


In [ ]:
# 11) Copy zip ra /kaggle/working để tải về
import shutil, os
for z in ["output.zip", "output_ner.zip"]:
    if os.path.exists(z): shutil.copy(z, f'/kaggle/working/{z}')
print("Tải output_ner.zip ở panel Output (bên phải) để nộp thi.")


## Ghi chú TRUNG THỰC
- Điểm `evaluate` là trên **dev synthetic (tự sinh)** → **lạc quan**, KHÔNG phải điểm thi thật.
- **Thuốc**: mã lấy từ **RxNav API** (build offline vào `data/kb/rxnorm_api.csv`); inference KHÔNG gọi API (đúng luật thi).
- **Bệnh (ICD)**: hiện match bằng seed tiếng Việt (fuzzy). Đã tải ICD-10 tiếng Anh (71k mã) làm code-space cho bước cross-lingual (SapBERT) — CHƯA bật.
- Nộp thi: **`output_ner.zip`** (hoặc `output.zip` nếu chưa train model).
- Cấu hình train ở cell 6 đã kiểm chứng KHÔNG OOM trên 16GB. Đổi sang `xlm-roberta-large` mà OOM thì giảm `--batch_size`.
